# CellAgent SCA Pipeline

统一入口：在 `cellagent` kernel 中一键执行预处理、QC report、通过 scGPT feature service 提取 encoder 特征、基于特征聚类、DE 分析。路径和参数从 `config/config.yaml` 读取。

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import time
import requests
import yaml

PROJECT_ROOT = Path('/root/wanghaoran/zxy/project/cellagent')
CONFIG = PROJECT_ROOT / 'config/config.yaml'
cfg = yaml.safe_load(CONFIG.read_text())
pipeline_cfg = cfg.get('pipeline', {})

INPUT_H5AD = Path(pipeline_cfg['input_path'])
OUTPUT_ROOT = Path(pipeline_cfg['output_root'])

PREPROCESSED_DIR = OUTPUT_ROOT / 'preprocessed'
QC_REPORT_DIR = OUTPUT_ROOT / 'qc_report'
FEATURE_DIR = OUTPUT_ROOT / 'features'
CLUSTER_DIR = OUTPUT_ROOT / 'clustering'
DE_DIR = OUTPUT_ROOT / 'de'
MULTIMODAL_PRIOR_DIR = OUTPUT_ROOT / 'multimodal_prior'
REASONING_DIR = OUTPUT_ROOT / 'reasoning'
FINAL_DIR = OUTPUT_ROOT / 'final'
MANIFEST = OUTPUT_ROOT / 'pipeline_manifest.json'

STEM = INPUT_H5AD.name[:-5] if INPUT_H5AD.name.endswith('.h5ad') else INPUT_H5AD.stem
PREPROCESSED_H5AD = PREPROCESSED_DIR / f'{STEM}_preprocessed.h5ad'
FEATURE_NPZ = FEATURE_DIR / f'{STEM}_preprocessed_cell_features.npz'
CLUSTERS_CSV = CLUSTER_DIR / f'{STEM}_preprocessed_cell_features_cell_clusters.csv'
CLUSTER_METRICS = CLUSTER_DIR / f'{STEM}_preprocessed_cell_features_clustering_metrics.json'

for path in [PREPROCESSED_DIR, QC_REPORT_DIR, FEATURE_DIR, CLUSTER_DIR, DE_DIR, MULTIMODAL_PRIOR_DIR, REASONING_DIR, FINAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'python': sys.executable,
    'config': str(CONFIG),
    'input': str(INPUT_H5AD),
    'output_root': str(OUTPUT_ROOT),
    'preprocessed': str(PREPROCESSED_H5AD),
    'feature_npz': str(FEATURE_NPZ),
    'clusters_csv': str(CLUSTERS_CSV),
    'feature_service': cfg.get('feature_service', {}),
}, ensure_ascii=False, indent=2))


{
  "python": "/root/miniconda3/envs/cellagent/bin/python3.10",
  "config": "/root/wanghaoran/zxy/project/cellagent/config/config.yaml",
  "input": "/data/bgi/data/projects/multimodal/RNA_data/cellxgene_disease/neural_disease/als/c48ec35b-a4e3-48a8-8407-87fe3f59caa2.h5ad",
  "output_root": "/home/qijinyin/wanghaoran/zxy/cellagent_output",
  "preprocessed": "/home/qijinyin/wanghaoran/zxy/cellagent_output/preprocessed/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed.h5ad",
  "feature_npz": "/home/qijinyin/wanghaoran/zxy/cellagent_output/features/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features.npz",
  "clusters_csv": "/home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_cell_clusters.csv",
  "feature_service": {
    "enabled": true,
    "url": "http://127.0.0.1:18080",
    "host": "0.0.0.0",
    "port": 18080,
    "gpu_id": 7,
    "max_workers": 1
  }
}


In [2]:
def run_cmd(cmd, skip_if_exists=None):
    if skip_if_exists is not None and Path(skip_if_exists).exists():
        print(f'[skip] exists: {skip_if_exists}')
        return
    full_cmd = [sys.executable] + [str(x) for x in cmd]
    print('[run]', ' '.join(full_cmd))
    subprocess.run(full_cmd, cwd=PROJECT_ROOT, check=True)

def require_exists(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    print(f'[ok] {path} ({path.stat().st_size} bytes)')

def require_feature_service():
    service_cfg = cfg.get('feature_service', {})
    if not service_cfg.get('enabled', False):
        raise RuntimeError('feature_service.enabled must be true for one-click cellagent notebook execution.')
    url = service_cfg.get('url', 'http://127.0.0.1:18080').rstrip('/')
    resp = requests.get(f'{url}/health', timeout=10)
    resp.raise_for_status()
    print('[feature-service]', json.dumps(resp.json(), ensure_ascii=False))
    return url


In [3]:
# 1. Preprocess + QC report. Runs in the current cellagent kernel environment.
if PREPROCESSED_H5AD.exists():
    print(f'[skip] exists: {PREPROCESSED_H5AD}')
else:
    run_cmd([
        'scripts/preprocess_external_data.py',
        '--input', INPUT_H5AD,
        '--output-dir', PREPROCESSED_DIR,
        '--qc-report-dir', QC_REPORT_DIR,
        '--config', CONFIG,
    ])
require_exists(PREPROCESSED_H5AD)

if not (QC_REPORT_DIR / 'preprocessing_summary.json').exists():
    run_cmd([
        'scripts/write_qc_report.py',
        '--input', PREPROCESSED_H5AD,
        '--output-dir', QC_REPORT_DIR,
    ])
require_exists(QC_REPORT_DIR / 'preprocessing_summary.json')
require_exists(QC_REPORT_DIR / 'qc_metrics_summary.csv')


[skip] exists: /home/qijinyin/wanghaoran/zxy/cellagent_output/preprocessed/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed.h5ad
[ok] /home/qijinyin/wanghaoran/zxy/cellagent_output/preprocessed/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed.h5ad (84009255 bytes)
[ok] /home/qijinyin/wanghaoran/zxy/cellagent_output/qc_report/preprocessing_summary.json (1551 bytes)
[ok] /home/qijinyin/wanghaoran/zxy/cellagent_output/qc_report/qc_metrics_summary.csv (690 bytes)


In [4]:
# 2. Cell encoder feature extraction through the scGPT feature service.
# The notebook stays in the cellagent environment; scGPT/torch dependencies live behind the service.
require_feature_service()
run_cmd([
    'scripts/request_feature_service.py',
    '--input', PREPROCESSED_H5AD,
    '--feature-dir', FEATURE_DIR,
    '--config', CONFIG,
], skip_if_exists=FEATURE_NPZ)
require_exists(FEATURE_NPZ)


[feature-service] {"status": "ok", "config_path": "/root/wanghaoran/zxy/project/cellagent/config/config.yaml", "gpu_id": 7, "cuda_visible_devices": "7", "jobs": {"queued": 0, "running": 0, "succeeded": 0, "failed": 0}}
[run] /root/miniconda3/envs/cellagent/bin/python3.10 scripts/request_feature_service.py --input /home/qijinyin/wanghaoran/zxy/cellagent_output/preprocessed/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed.h5ad --feature-dir /home/qijinyin/wanghaoran/zxy/cellagent_output/features --config /root/wanghaoran/zxy/project/cellagent/config/config.yaml
[feature-service] job=ba48fb4ada4e48c4861127976f342cf8 status=running


[feature-service] job=ba48fb4ada4e48c4861127976f342cf8 status=running


{
  "job_id": "ba48fb4ada4e48c4861127976f342cf8",
  "status": "succeeded",
  "created_at": 1778572806.007736,
  "updated_at": 1778572812.7260282,
  "request": {
    "input_h5ad": "/home/qijinyin/wanghaoran/zxy/cellagent_output/preprocessed/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed.h5ad",
    "feature_dir": "/home/qijinyin/wanghaoran/zxy/cellagent_output/features",
    "model_path": null,
    "vocab_path": null,
    "feature_layer": null,
    "gene_id_column": null,
    "n_genes": null,
    "batch_size": 128,
    "output_format": null,
    "device": null,
    "scgpt_root": null
  },
  "feature_path": "/home/qijinyin/wanghaoran/zxy/cellagent_output/features/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features.npz",
  "error": null,
  "traceback": null
}
[ok] /home/qijinyin/wanghaoran/zxy/cellagent_output/features/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features.npz (25636718 bytes)


In [5]:
# 3. Leiden clustering on extracted cell encoder features.
run_cmd([
    'scripts/cluster_cell_features.py',
    '--features', FEATURE_NPZ,
    '--output-dir', CLUSTER_DIR,
    '--config', CONFIG,
], skip_if_exists=CLUSTERS_CSV)
require_exists(CLUSTERS_CSV)
require_exists(CLUSTER_METRICS)
print(json.dumps(json.loads(CLUSTER_METRICS.read_text()), ensure_ascii=False, indent=2)[:2000])


[run] /root/miniconda3/envs/cellagent/bin/python3.10 scripts/cluster_cell_features.py --features /home/qijinyin/wanghaoran/zxy/cellagent_output/features/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features.npz --output-dir /home/qijinyin/wanghaoran/zxy/cellagent_output/clustering --config /root/wanghaoran/zxy/project/cellagent/config/config.yaml


/root/wanghaoran/zxy/project/cellagent/src/tools/clustering.py:121: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=resolution, key_added=key, random_state=cfg.random_state)


{
  "h5ad": "/home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_clustered.h5ad",
  "clusters_csv": "/home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_cell_clusters.csv",
  "metrics_json": "/home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_clustering_metrics.json"
}


[ok] /home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_cell_clusters.csv (920246 bytes)
[ok] /home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_clustering_metrics.json (1172 bytes)
{
  "feature_path": "/home/qijinyin/wanghaoran/zxy/cellagent_output/features/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features.npz",
  "n_cells": 17799,
  "n_features": 768,
  "n_pcs": 30,
  "n_neighbors": 30,
  "metric": "cosine",
  "resolutions": [
    0.25,
    0.5,
    0.75,
    1.0
  ],
  "choose_by": "modularity",
  "min_clusters": 5,
  "max_clusters": 15,
  "best_resolution": 0.75,
  "best_key": "leiden_r0_75",
  "best_n_clusters": 12,
  "best_modularity": 0.83177582038972,
  "metrics": [
    {
      "resolution": 0.25,
      "key": "leiden_r0_25",
      "n_clusters": 6,
      "modularity": 0.7446033620935169,
      "cluster_count_pass": true
    

In [6]:
# 4. DE analysis by cluster. Saves up to config.de_analysis.top_k significant genes per cluster.
de_summary = DE_DIR / 'de_summary.csv'
run_cmd([
    'scripts/run_de_analysis.py',
    '--preprocessed', PREPROCESSED_H5AD,
    '--clusters', CLUSTERS_CSV,
    '--output-dir', DE_DIR,
    '--config', CONFIG,
], skip_if_exists=de_summary)
require_exists(de_summary)
print(de_summary.read_text().splitlines()[:12])


[run] /root/miniconda3/envs/cellagent/bin/python3.10 scripts/run_de_analysis.py --preprocessed /home/qijinyin/wanghaoran/zxy/cellagent_output/preprocessed/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed.h5ad --clusters /home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_cell_clusters.csv --output-dir /home/qijinyin/wanghaoran/zxy/cellagent_output/de --config /root/wanghaoran/zxy/project/cellagent/config/config.yaml


{
  "cluster_0": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_0_vs_all.csv",
  "cluster_1": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_1_vs_all.csv",
  "cluster_2": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_2_vs_all.csv",
  "cluster_3": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_3_vs_all.csv",
  "cluster_4": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_4_vs_all.csv",
  "cluster_5": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_5_vs_all.csv",
  "cluster_6": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_6_vs_all.csv",
  "cluster_7": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_7_vs_all.csv",
  "cluster_8": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_8_vs_all.csv",
  "cluster_9": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_9_vs_all.csv",
  "cluster_10": "/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_10_vs_all.csv",
  "cluster_11": "

[ok] /home/qijinyin/wanghaoran/zxy/cellagent_output/de/de_summary.csv (1009 bytes)
['cluster,n_cells,n_genes_saved,csv', '0,2198,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_0_vs_all.csv', '1,2092,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_1_vs_all.csv', '2,1913,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_2_vs_all.csv', '3,1762,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_3_vs_all.csv', '4,1706,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_4_vs_all.csv', '5,1626,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_5_vs_all.csv', '6,1590,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_6_vs_all.csv', '7,1204,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_7_vs_all.csv', '8,1101,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_8_vs_all.csv', '9,1032,30,/home/qijinyin/wanghaoran/zxy/cellagent_output/de/cluster_9_vs_all.csv', '10,1029,30,/home/qijinyin/wanghaoran/z

In [7]:
# 5. Write the stable artifact manifest for downstream multimodal prior / reasoning / judge notebooks.
run_cmd([
    'scripts/write_pipeline_manifest.py',
    '--config', CONFIG,
    '--output', MANIFEST,
    '--preprocessed-h5ad', PREPROCESSED_H5AD,
    '--qc-report-dir', QC_REPORT_DIR,
    '--feature-npz', FEATURE_NPZ,
    '--clusters-csv', CLUSTERS_CSV,
    '--clustering-metrics-json', CLUSTER_METRICS,
    '--de-summary-csv', de_summary,
    '--de-dir', DE_DIR,
    '--reasoning-dir', REASONING_DIR,
    '--final-dir', FINAL_DIR,
])
require_exists(MANIFEST)
print(json.dumps(json.loads(MANIFEST.read_text()), ensure_ascii=False, indent=2)[:3000])


[run] /root/miniconda3/envs/cellagent/bin/python3.10 scripts/write_pipeline_manifest.py --config /root/wanghaoran/zxy/project/cellagent/config/config.yaml --output /home/qijinyin/wanghaoran/zxy/cellagent_output/pipeline_manifest.json --preprocessed-h5ad /home/qijinyin/wanghaoran/zxy/cellagent_output/preprocessed/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed.h5ad --qc-report-dir /home/qijinyin/wanghaoran/zxy/cellagent_output/qc_report --feature-npz /home/qijinyin/wanghaoran/zxy/cellagent_output/features/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features.npz --clusters-csv /home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_cell_clusters.csv --clustering-metrics-json /home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_clustering_metrics.json --de-summary-csv /home/qijinyin/wanghaoran/zxy/cellagent_output/de/de_summary.csv --de-dir /h

[ok] /home/qijinyin/wanghaoran/zxy/cellagent_output/pipeline_manifest.json (6367 bytes)
{
  "run_id": "c48ec35b-a4e3-48a8-8407-87fe3f59caa2",
  "input_h5ad": "/data/bgi/data/projects/multimodal/RNA_data/cellxgene_disease/neural_disease/als/c48ec35b-a4e3-48a8-8407-87fe3f59caa2.h5ad",
  "output_root": "/home/qijinyin/wanghaoran/zxy/cellagent_output",
  "config_path": "/root/wanghaoran/zxy/project/cellagent/config/config.yaml",
  "preprocessed_h5ad": "/home/qijinyin/wanghaoran/zxy/cellagent_output/preprocessed/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed.h5ad",
  "qc_report_dir": "/home/qijinyin/wanghaoran/zxy/cellagent_output/qc_report",
  "feature_npz": "/home/qijinyin/wanghaoran/zxy/cellagent_output/features/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features.npz",
  "clusters_csv": "/home/qijinyin/wanghaoran/zxy/cellagent_output/clustering/c48ec35b-a4e3-48a8-8407-87fe3f59caa2_preprocessed_cell_features_cell_clusters.csv",
  "clustering_metrics_json": "/home/qijinyin/w